# BHJet spectrum and NGC 4261 data

This exercise uses the installed `bhjet` package and the versioned NGC 4261 data stored in this repository. No local file paths or downloads are required.


In [ ]:
from pathlib import Path

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table

from bhjet.gammapy import BHJetSpectralModel

def find_repository_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'examples' / 'gammapy' / 'data' / 'ngc4261' / 'ngc4261_dnde.csv').is_file():
            return candidate
    raise FileNotFoundError('Open this notebook from a complete modBHJet checkout.')

repository_root = find_repository_root()
data_file = repository_root / 'examples' / 'gammapy' / 'data' / 'ngc4261' / 'ngc4261_dnde.csv'
data_file


## Load the bundled public data

The file has reference energy, differential number flux, and its uncertainty. The units are assigned explicitly so that the calculation remains clear and portable.


In [ ]:
data = Table.read(data_file, format='ascii.csv', delimiter=' ')
data['e_ref'].unit = u.eV
data['dnde'].unit = u.Unit('cm-2 s-1 eV-1')
data['dnde_err'].unit = u.Unit('cm-2 s-1 eV-1')
data[:5]


In [ ]:
data_energy = data['e_ref'].quantity
data_sed = (data_energy.to(u.erg) ** 2 * data['dnde'].quantity.to('cm-2 s-1 erg-1')).to('erg cm-2 s-1')
data_sed_error = (data_energy.to(u.erg) ** 2 * data['dnde_err'].quantity.to('cm-2 s-1 erg-1')).to('erg cm-2 s-1')

fig, ax = plt.subplots()
ax.errorbar(data_energy.to_value(u.eV), data_sed.value, yerr=data_sed_error.value, fmt='o', label='NGC 4261')
ax.set(xscale='log', yscale='log', xlabel='Observed energy [eV]', ylabel=r'$E^2 dN/dE$ [erg cm$^{-2}$ s$^{-1}$]')
ax.grid(alpha=0.3)
ax.legend()


## Calculate a BHJet spectrum through Gammapy

`BHJetSpectralModel` exposes the BHJet calculation as a Gammapy spectral model. Its calculation is cached while the parameters remain unchanged.


In [ ]:
model = BHJetSpectralModel()
model.jet_power_eddington.value = 2e-3
model.z_dissipation.value = 100.0
model.index_injected_electrons.value = 1.8
model.parameters.to_table()


In [ ]:
energy = np.geomspace(1e-6, 1e12, 120) * u.eV
model_flux = model(energy)
model_sed = (energy.to(u.erg) ** 2 * model_flux).to('erg cm-2 s-1')

fig, ax = plt.subplots()
ax.errorbar(data_energy.to_value(u.eV), data_sed.value, yerr=data_sed_error.value, fmt='o', label='NGC 4261 data')
ax.loglog(energy.to_value(u.eV), model_sed.value, label='BHJet model')
ax.set(xlabel='Observed energy [eV]', ylabel=r'$E^2 dN/dE$ [erg cm$^{-2}$ s$^{-1}$]')
ax.grid(alpha=0.3)
ax.legend()


## Try it

Change `jet_power_eddington`, `z_dissipation`, or `index_injected_electrons` above and re-run the final cell. Then compare which part of the spectrum moves most strongly.
